# SOMA AND NEURITE FILTERING

-------

## OBJECTIVE:
In this notebook, the logic for filtering the soma from the neurites in is outlined. This filtering requires the segmentation of the cell mask to already have occured and is completely automatic. Additionally, at the end of the notebook, functions for batch processing this filtering is outlined.

If using this notebook solely for batch processing, the following sections of code must be run in the following order prior to running the batch processing section:

1. Imports
2. Define Function to find Radii
3. Define Function that segments soma from neurites and cleans image

## SUMMARY OF WORKFLOW STEPS:
 - Inputs
    - Get and load pre-segmented image
 - Pre-Processing
    - Determine radius to filter neurites from soma
 - Core-Processing
    - Filter the soma from the neurites using erosion and dilating techniques
 - Post-Processing
    - Ensure soma area looks correct
 - Export
    - Export the newly filtered image

## IMPORTS:

THIS

In [1]:
from pathlib import Path
import os, sys
from infer_subc.core.file_io import (list_image_files, 
                                     read_czi_image, 
                                     import_inferred_organelle, 
                                     export_inferred_organelle)
import napari
from skimage.morphology import binary_opening, binary_dilation, binary_erosion
from scipy.ndimage import zoom
from infer_subc.core.img import *

viewer = napari.Viewer()

-------
## Soma Filter Workflow

### INPUTS
Load the pre-segmented cell mask image. The pre-segmented image will be used to determine where the soma an neurites are located based on the diameters of the sections of the object.

#### User Inputs

##### Image Path
Here, the user should edit any of the values to correctly access the folder containing the __RAW__ image files. The naming of this file will then be used to collect the corresponding segmented cell mask files. To correctly ensure the cell mask files are found, changing the __seg_type__ variable to the same value as is found at the end of the cell mask file names is required.

In [ ]:
test_img_n = 0

in_data_path = Path("Z:/Cohen Lab/Maria Clara/2_Lab data/1_Multispectral data/2023/112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2/Deconvolved iN day28 images water Cp/tiff")
seg_data_path = Path("Z:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/Segmentation iNday28/Cell mask 8bit")
out_data_path = Path("Z:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/Segmentation iNday28/soma neurites 8bit")
seg_suffix = "-"
seg_type = "cell"
im_type = ".tiff"


img_file_list = list_image_files(in_data_path,im_type)
test_img_name = img_file_list[test_img_n]

if not Path.exists(out_data_path):
    Path.mkdir(out_data_path)
    print(f"making {out_data_path}")

In [ ]:
img_data,meta_dict = read_czi_image(test_img_name)
cell_mask = import_inferred_organelle(name=seg_type,meta_dict=meta_dict,out_data_path=seg_data_path,file_type=im_type)
channel_names = meta_dict['name']
img = meta_dict['metadata']['aicsimage']
scale = meta_dict['scale']
channel_axis = meta_dict['channel_axis']

#### Visualize the inputs
To better understand the image, the raw organelle image and the segmented organelle image are added to the napari viewer using the below lines of code.

In [ ]:
viewer.add_image(img_data, name="Raw Image", scale=scale)
viewer.add_labels(cell_mask, name="Raw Mask", scale=scale)

### PRE-PROCESSING

#### Rescale Image to enable the code to run
Due to the large memory requirements, rescaling the image to half its size greatly reduces the complexity of the tasks to be performed.

In [ ]:
cell_mask_resize = zoom(cell_mask.copy(), (1, 0.5, 0.5))

#### Find Radii
First, we must determine the correct radius to use to separate the soma from the neurites. This process is completely automated and relies on a sorting algorithm to determine what radius is ideal.

##### Find Range of Possible Radii
The range of the possible radii is defined as 1/4 of the size of the image.

In [ ]:
zz, yy, xx = np.shape(cell_mask_resize)
rad_range = [i+1 for i in list(range(yy // 4))]

##### Determine Soma Radii
Using the range of possible radii, the program takes the middle-most radius of the range, and checks whether it is too large or to small. If the radius is too small, all radii smaller than it are removed as possibilities. If the radius is too large, all radii larger than it are removed from possibilities. This process occurs until there is only 1 or 2 radii remaining.

In [ ]:
while len(rad_range) > 2:
    rad = rad_range[((len(rad_range)) // 2)]
    print(f"Trying radius of {rad}")
    edge = disk(rad//4)
    middle = disk(rad)
    w = (np.shape(middle)[0] - np.shape(edge)[0])//2
    edge = np.pad(edge, ((w,w),(w,w)), mode = 'constant', constant_values=0)
    fp = np.stack((edge, middle, edge))
    if np.array_equal((binary_erosion(cell_mask_resize, fp)), np.zeros_like(cell_mask_resize)):
        rad_range = rad_range[:(rad_range.index(rad))]
        print(f"{rad} is too large")
    else:
        rad_range = rad_range[(rad_range.index(rad)+1):]
        print(f"{rad} is too small")
    print(f"{len(rad_range)} possible radii remaining")

##### Determine Optimal Radii
After finding the radus of the soma, the optimal radius for the removal of neurites is defined as 1/4th the radius of the soma. 

NOTE: Because the image has been scaled down to half while finding an optimal radius, the output is only divided by 2, but this will still be 1/4th of the radius of the soma.

In [ ]:
if len(rad_range) == 1:
    opti_rad = rad_range[0]//2
elif len(rad_range) == 2:
    opti_rad = rad_range[1]//2

#### Define Function to Find Radii

Find_Rad will take a pre-zoomed image and output the radius for the neurites based on this image. Rather than using a while loop, this function acts as a recursive function to improve memory usage and speed of the computational tasks.

THIS

In [2]:
def find_rad(in_seg: np.ndarray, rad_range: list = []) -> int:
    if rad_range == []:
        zz, yy, xx = np.shape(in_seg)
        rad_range = [i+1 for i in list(range(yy // 4))]
    if len(rad_range) == 1:
        return rad_range[0]//2
    elif len(rad_range) == 2:
        return rad_range[1]//2
    rad = rad_range[((len(rad_range)) // 2)]
    print(f"Trying radius of {rad}")
    edge = disk(rad//4)
    middle = disk(rad)
    w = (np.shape(middle)[0] - np.shape(edge)[0])//2
    edge = np.pad(edge, ((w,w),(w,w)), mode = 'constant', constant_values=0)
    fp = np.stack((edge, middle, edge))
    if np.array_equal((binary_erosion(in_seg, fp)), np.zeros_like(in_seg)):
        rad_range = rad_range[:(rad_range.index(rad))]
        print(f"{rad} is too large")
    else:
        rad_range = rad_range[(rad_range.index(rad)+1):]
        print(f"{rad} is too small")
    print(f"{len(rad_range)} possible radii remaining")
    return find_rad(in_seg, rad_range)

#### Check that both radii are the same:

In [ ]:
print(f"The statement that both radii are the same is {find_rad(in_seg=cell_mask_resize) == opti_rad}")

### CORE-PROCESSING


#### Filter Soma from Neurites
Using the optimal radius to remove the neurites with, the cell mask is eroded and then dilated back to its original size. When a process is eroded enough, it will disappear, this results in the removal of sections of the mask that are small enough in diameter (the neurites).



##### Develop the Footprint for the Erosion/Dilation
Here, we use the optimal radius to create a footprint for both the erosion and dilation. This footprint is a 3 dimensional saucer that is nearly flat.

In [ ]:
edge = disk(opti_rad//2)
middle = disk(opti_rad)
w = (np.shape(middle)[0] - np.shape(edge)[0])//2
edge = np.pad(edge, ((w,w),(w,w)), mode = 'constant', constant_values=0)
fp = np.stack((edge, middle, edge))

##### Perform the Erosion/Dilation
Using the opening function to erode the image and then dilate it by the same radius, we remove the neurites and recover the soma.

In [ ]:
neurites_removed = binary_opening(cell_mask, fp)

### POST-PROCESSING

#### Resizing Image to original size

#### Image Cleaning
Due to the nature of the process of eroding and dilating an image, some minor pixels will be missing from the soma. This is fixed by further expanding the image and masking it with the original cell mask. 

In [ ]:
soma = binary_dilation(neurites_removed, footprint=ball((opti_rad)//2)) * cell_mask

#### Find the Neurites
Neurites can now be defined as anywhere in the cell mask that is not present in the soma.

In [ ]:
neurites = np.invert(soma.astype(bool), dtype=bool) * cell_mask

#### Fix any outcrops and loose specks
Small "spikes" coming out of the soma are often removed in this process. The below code ensures that these "spikes" are still considered part of the soma rather than considered as neurites. This process works by finding small enough outcrops/spikes and adding them to the soma image, then ensuring that only objects connected to the soma image are considered the soma and the others are removed. Afterwards, the neurites are considered anything that is not chosen. Finally, the neurites once again get the small objects removed as these objects will not be connected to the soma and are likely just loose specks.

In [ ]:
neurites = size_filter_linear_size(img=label(neurites), min_size=(opti_rad//2), method='3D')
soma = label(np.invert(neurites.astype(bool), dtype=bool) * cell_mask)
soma[soma!=np.bincount(np.ravel(soma)[np.ravel(soma)!= 0]).argmax()] = 0
neurites = np.invert(soma.astype(bool), dtype=bool) * cell_mask
neurites = size_filter_linear_size(img=label(neurites), min_size=(opti_rad//2), method='3D')
soma = label(soma)
neurites = label(neurites)

#### Visualize Results
Prior to exporting the soma and neurites, we will visualize the results in napari.

In [ ]:
viewer.add_labels(soma, name="Soma", scale=scale)
viewer.add_labels(neurites, name="Neurites", scale=scale)

#### Define function that segments soma from neurites and cleans the image

THIS

In [3]:
def soma_neurite_filter(in_seg: np.ndarray):
    cell_mask = zoom(in_seg, (1, 0.5, 0.5))
    rad = find_rad(cell_mask)
    edge = disk(rad//2)
    middle = disk(rad)
    w = (np.shape(middle)[0] - np.shape(edge)[0])//2
    edge = np.pad(edge, ((w,w),(w,w)), mode = 'constant', constant_values=0)
    fp = np.stack((edge, middle, edge))
    nr1 = binary_opening(in_seg, fp)
    soma = binary_dilation(nr1, footprint=ball((rad)//2)) * in_seg
    neurites = np.invert(soma.astype(bool), dtype=bool) * in_seg
    neurites = size_filter_linear_size(img=label(neurites), min_size=(rad//2), method='3D')
    soma = label(np.invert(neurites.astype(bool), dtype=bool) * in_seg)
    soma[soma!=np.bincount(np.ravel(soma)[np.ravel(soma)!= 0]).argmax()] = 0
    neurites = np.invert(soma.astype(bool), dtype=bool) * in_seg
    neurites = size_filter_linear_size(img=label(neurites), min_size=(rad//2), method='3D')
    return label(soma), label(neurites)

#### Check that both sets of outputs are the same

In [ ]:
som, neu = soma_neurite_filter(cell_mask)
print(f"The statement that both outputs are the same is {((np.array_equal((som), (soma))) and (np.array_equal((neu), (neurites))))}")

### Export
Here, we will export both the soma and neurites into the same folder locations. The suffix for the soma file will become "soma" and the suffix for the neurites file will become "neurites".

In [ ]:
out_file_n = export_inferred_organelle(soma, "soma", meta_dict, out_data_path)
out_file_n = export_inferred_organelle(neurites, "neurites", meta_dict, out_data_path)

-------
## Batch Processing Filter
This section of the notebook enables batch processing of the soma and neurite filter to be performed on multiple cells at once. For this code to work, the __find_rad__ and __soma_neurite_filter__ functions must already be run. After these three functions are run, the code in this section will work properly. 

### User Inputs

#### Existing File Paths
To make the filter batch processable, we must create a list of all the images rather than just taking in one image at a time. To do this, we need two separate file paths: a raw file path, and a segmentation file path. These are the locations of the folders that contain all the unmixed images, and the location of the folder that contains the segmented organelle images. The computer must also be told the suffix for the mask file type, this suffix is usually "cell". Finally, the computer must also be told the type of file that is being searched for--usually, this file type is ".tiff".

In [4]:
raw_path=Path("Z:/Cohen Lab/Maria Clara/2_Lab data/1_Multispectral data/2023/112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2/Deconvolved iN day21 images 082024/tiff")
seg_path=Path("Z:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/Segmentation iNday21/InferSubC_iNday21/cell edited")
mask_suffix="cell"
file_type=".tiff"


#### Output File Path
You will also need a path for the folder where the newly declumped images will be exported to.

In [5]:
out_path=Path("Z:/Cohen Lab/Maria Clara/2_Lab data/9_Napari/Segmentation iNday21/soma neurite 8bit")

### Batch Process Function
Here, the function to perform the batch processing is defined. This function searches through each file in the unmixing folder, and searches through the segmentation folder for the corresponding file, and runs the declumping protocol on those files. The resulting declumped organelle image is output into the folder __"output file path"__ is defined as.

In [6]:
def filter_batch_process(raw_path: Union[Path,str],
                         seg_path: Union[Path,str],
                         out_path: Union[Path,str],
                         mask_suffix: str,
                         file_type: str):
    filtered_list = []
    img_file_list = list_image_files(raw_path, file_type)
    for img_f in img_file_list:
        print(f"Applying filter to {img_f}...")
        img_data, meta_dict = read_czi_image(img_f)
        cell_mask = import_inferred_organelle(name=mask_suffix,meta_dict=meta_dict,
                                              out_data_path=seg_path,file_type=file_type)
        soma, neurites = soma_neurite_filter(cell_mask)
        out_file_n1 = export_inferred_organelle(soma, "soma", meta_dict, out_path)
        out_file_n2 = export_inferred_organelle(neurites, "neurites", meta_dict, out_path)
        filtered_list.append((out_file_n1, out_file_n2))
        viewer.add_image(img_data, name="Raw", scale=meta_dict['scale'])
        viewer.add_labels(soma, name="Soma", scale = meta_dict['scale'])
        viewer.add_labels(neurites, name="Neurites", scale=meta_dict['scale'])
        print(f"Completed {img_file_list.index(img_f)+1} out of {len(img_file_list)+1} images.")
    return filtered_list

### Running Batch Processing
This code enables the above batch process function to run. It requires no user inputs as those inputs are called upon in the below chunk of code.

In [7]:
declumped_list = filter_batch_process(raw_path=raw_path,
                                      seg_path=seg_path,
                                      out_path=out_path,
                                      mask_suffix=mask_suffix,
                                      file_type=file_type)

Applying filter to Z:\Cohen Lab\Maria Clara\2_Lab data\1_Multispectral data\2023\112023_MSi08-L_Neuronal Differentiation - iPSCs-hNGN2\Deconvolved iN day21 images 082024\tiff\02082024_MSi08L_iN_Day21_BR3a_N23_Unmixing_0_cmle.ome.tiff...
loaded  inferred 3D `cell`  from Z:\Cohen Lab\Maria Clara\2_Lab data\9_Napari\Segmentation iNday21\InferSubC_iNday21\cell edited 
Trying radius of 50
50 is too large
49 possible radii remaining
Trying radius of 25
25 is too small
24 possible radii remaining
Trying radius of 38
38 is too small
11 possible radii remaining
Trying radius of 44
44 is too small
5 possible radii remaining
Trying radius of 47
47 is too small
2 possible radii remaining
saved file: 02082024_MSi08L_iN_Day21_BR3a_N23_Unmixing_0_cmle.ome-soma
saved file: 02082024_MSi08L_iN_Day21_BR3a_N23_Unmixing_0_cmle.ome-neurites
Completed 1 out of 2 images.
